# Advanced Usage Tutorial

This tutorial covers advanced features and use cases for nnunetsegmentator.

## Multi-Modal Segmentation

Segment multiple modalities (e.g., T1, T2, FLAIR) simultaneously:

In [ ]:
from nnunetsegmentator import SegmentationOrchestrator, Config
import nibabel as nib
import numpy as np

# Initialize orchestrator
orchestrator = SegmentationOrchestrator(
    task_name='gtrc',
    config=Config(use_gpu=True, num_workers=4)
)

# Load multiple modalities
modalities = {
    't1': 'path/to/t1.nii.gz',
    't2': 'path/to/t2.nii.gz',
    'flair': 'path/to/flair.nii.gz'
}

# Segment with multi-modal input
result = orchestrator.segment(
    input_data=modalities,
    output_path='path/to/output.nii.gz',
    return_labels=True,
    compute_metrics=True
)

print(f"Multi-modal segmentation complete!")
print(f"Output shape: {result.segmentation.GetSize()}")

## Batch Processing

Process multiple images efficiently:

In [ ]:
import os
from pathlib import Path
from nnunetsegmentator import SegmentationOrchestrator, Config

# Initialize orchestrator
orchestrator = SegmentationOrchestrator(
    task_name='gtrc',
    config=Config(use_gpu=True, num_workers=4, batch_size=4)
)

# Define input and output directories
input_dir = Path('path/to/input/images')
output_dir = Path('path/to/output/segmentations')
output_dir.mkdir(exist_ok=True)

# Process all files in directory
input_files = list(input_dir.glob('*.nii.gz'))

for i, input_file in enumerate(input_files, 1):
    print(f"Processing file {i}/{len(input_files)}: {input_file.name}")
    
    output_file = output_dir / f"{input_file.stem}_seg.nii.gz"
    
    result = orchestrator.segment(
        input_data=str(input_file),
        output_path=str(output_file),
        return_labels=False,
        compute_metrics=False
    )
    
    print(f"  → Saved to: {output_file}")

print(f"\nBatch processing complete! Processed {len(input_files)} files.")

## Custom Pipeline

Create a custom preprocessing and post-processing pipeline:

In [ ]:
from nnunetsegmentator import SegmentationOrchestrator, Config
from nnunetsegmentator.pipeline.base import BasePreprocessor, BasePostprocessor

# Define custom preprocessor
class CustomPreprocessor(BasePreprocessor):
    def preprocess(self, image):
        # Custom preprocessing logic
        # Example: Apply specific normalization
        image_array = image.numpy()
        image_array = (image_array - image_array.mean()) / (image_array.std() + 1e-8)
        return image_array

# Define custom postprocessor
class CustomPostprocessor(BasePostprocessor):
    def postprocess(self, segmentation):
        # Custom post-processing logic
        # Example: Apply specific filtering
        # Your custom logic here
        return segmentation

# Use custom pipeline
orchestrator = SegmentationOrchestrator(
    task_name='gtrc',
    config=Config(use_gpu=True)
)

# Note: Custom pipeline integration requires modifying the task configuration
# See Custom Tasks tutorial for details

## Metrics and Evaluation

Compute segmentation metrics:

In [ ]:
from nnunetsegmentator import SegmentationOrchestrator, Config

# Initialize orchestrator
orchestrator = SegmentationOrchestrator(
    task_name='gtrc',
    config=Config(use_gpu=True)
)

# Segment with metrics computation
result = orchestrator.segment(
    input_data='path/to/image.nii.gz',
    output_path='path/to/output.nii.gz',
    return_labels=True,
    compute_metrics=True
)

# Access detailed metrics
if result.metrics:
    print("Segmentation Metrics:")
    for metric_name, metric_value in result.metrics.items():
        print(f"  {metric_name}: {metric_value}")
    
    # Access specific metrics
    if 'volume_mm3' in result.metrics:
        print(f"\nTotal Volume: {result.metrics['volume_mm3']:.2f} mm³")
    
    if 'surface_dice' in result.metrics:
        print(f"Surface Dice: {result.metrics['surface_dice']:.4f}")

## Performance Optimization

Optimize segmentation for speed and memory:

In [ ]:
from nnunetsegmentator import Config

# Optimize for speed
config_fast = Config(
    use_gpu=True,
    num_workers=8,
    batch_size=8,
    patch_size=[96, 96, 48],  # Smaller patches
    overlap_factor=0.5
)

# Optimize for accuracy
config_accurate = Config(
    use_gpu=True,
    num_workers=4,
    batch_size=2,
    patch_size=[160, 160, 80],  # Larger patches
    overlap_factor=0.7,
    tta=True  # Test-time augmentation
)

# Optimize for memory
config_memory = Config(
    use_gpu=True,
    num_workers=2,
    batch_size=1,
    patch_size=[128, 128, 64],
    overlap_factor=0.5,
    mixed_precision=True  # Use FP16
)

## Error Handling

Handle potential errors gracefully:

In [ ]:
from nnunetsegmentator import SegmentationOrchestrator, Config
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize orchestrator
orchestrator = SegmentationOrchestrator(
    task_name='gtrc',
    config=Config(use_gpu=True)
)

try:
    result = orchestrator.segment(
        input_data='path/to/image.nii.gz',
        output_path='path/to/output.nii.gz',
        return_labels=True,
        compute_metrics=True
    )
    
    logger.info("Segmentation completed successfully")
    
except FileNotFoundError as e:
    logger.error(f"Input file not found: {e}")
except RuntimeError as e:
    logger.error(f"Runtime error during segmentation: {e}")
except Exception as e:
    logger.error(f"Unexpected error: {e}")

## Next Steps

- Learn about [Custom Tasks](custom_tasks.ipynb) for building your own pipelines
- Check the [API Reference](../reference/api-reference.md) for detailed method documentation
- Explore [Examples](../examples/) for more use cases